# Phase C: Chronological Baseline and Initial ML Modeling

This notebook uses only the approved Phase B features and never shuffles rows. Model selection is based on expanding-window validation within the first 400 observations; the final 100 observations are evaluated only after selection.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().parent
if not (project_root / 'src').exists():
    project_root = Path.cwd()
sys.path.insert(0, str(project_root))

from src.data_preprocessing import FEATURE_COLUMNS, TARGET_COLUMN, chronological_split, load_and_validate_data
from src.train import build_model_pipelines, fit_and_test_models, validate_models

data = load_and_validate_data(project_root / 'EV_Battery_Charging_TR_Dataset_with_Notes (1).csv')
development_data, final_test_data = chronological_split(data)
X_development = development_data[FEATURE_COLUMNS]
y_development = development_data[TARGET_COLUMN]
X_test = final_test_data[FEATURE_COLUMNS]
y_test = final_test_data[TARGET_COLUMN]
print('Approved features:', FEATURE_COLUMNS)
print('Development:', development_data['Timestamp'].min(), 'to', development_data['Timestamp'].max())
print('Untouched test:', final_test_data['Timestamp'].min(), 'to', final_test_data['Timestamp'].max())

In [ ]:
pipelines = build_model_pipelines()
validation_results = validate_models(pipelines, X_development, y_development)
selected_model = validation_results.iloc[0]['model']
print('Selected on validation MAE only:', selected_model)
validation_results

## Final chronological test evaluation

Run this only after reviewing the validation comparison. Its result must not be used to change model settings or select a model.

In [ ]:
fitted_models, test_results, predictions = fit_and_test_models(
    pipelines, X_development, y_development, X_test, y_test
)
test_results

The reproducible command `python -m src.train` additionally writes the comparison CSV and required figures. It does not tune on or re-use the final test period.